# Giorno 4 — Ensemble, validazione incrociata, ottimizzazione e Pipeline

> Linguaggio degli esempi: Python 3 (scikit-learn ≥ 1.4, NumPy, pandas, matplotlib, SciPy). Ogni modulo segue la struttura: Spiegazione teorica → Esempi teorici di utilizzo → Esempi di codice → Esercizi.
> Prerequisiti dai giorni precedenti: alberi di decisione e loro instabilità, overfitting e underfitting, validation set, regola d'oro del preprocessing, iperparametri dei modelli (k, max_depth, C, alpha, gamma).

# 4.1 Metodi ensemble: Random Forest e Boosting

## Spiegazione teorica

**Idea:** un **ensemble** combina le previsioni di **molti modelli** per ottenere un modello finale migliore di ciascuno di essi. È la "saggezza della folla": se ogni modello sbaglia in modo diverso, gli errori tendono ad annullarsi.

**Bagging (Bootstrap AGGregatING)**
- **Bootstrap:** si crea un nuovo dataset estraendo campioni **con reinserimento** dal training (lo stesso campione può uscire più volte, altri non escono mai). Ogni dataset bootstrap ha la stessa dimensione dell'originale ma contenuto diverso.
- Si addestra un modello su ogni dataset bootstrap, **in parallelo** e in modo indipendente.
- Si **aggregano** le previsioni: voto di maggioranza (classificazione) o media (regressione).
- Effetto: riduce la **varianza** → ideale per modelli instabili come gli alberi profondi.

**Random Forest = bagging di alberi + casualità sulle feature**
- Ogni albero è addestrato su un dataset bootstrap **e**, a ogni split, può scegliere solo tra un **sottoinsieme casuale di feature** (`max_features`).
- Così gli alberi sono diversi tra loro (decorrelati) e i loro errori si compensano meglio.
- Iperparametri principali:
  - `n_estimators` (default 100): numero di alberi; più alberi = risultato più stabile, ma più tempo. Aumentarli non causa overfitting.
  - `max_features`: feature valutate a ogni split; default `"sqrt"` (radice del numero di feature) per la classificazione. Il valore `"auto"` usato in vecchi tutorial (e nel PDF) è stato **rimosso**.
  - `max_depth`, `min_samples_leaf`, `criterion`: come negli alberi singoli.
  - `n_jobs=-1`: addestra gli alberi in parallelo su tutti i processori.
  - `oob_score=True`: ogni albero non ha visto circa un terzo dei campioni (quelli rimasti fuori dal bootstrap, **out-of-bag**); valutandoli si ottiene una stima di validazione "gratuita".

**Boosting**
- I modelli vengono addestrati **in sequenza**: ogni nuovo modello si concentra sugli **errori** di quelli precedenti.
- Si usano modelli deboli (**weak learner**): tipicamente alberi molto piccoli, poco meglio del caso da soli.
- Effetto: riduce soprattutto il **bias**; molto potente, ma può andare in overfitting se si esagera.
- **AdaBoost:** dopo ogni modello aumenta il **peso** dei campioni classificati male, così il modello successivo si concentra su di essi.
- **Gradient Boosting:** ogni nuovo albero impara a prevedere gli **errori residui** del modello attuale, e la sua correzione viene sommata.
  - `n_estimators`: numero di alberi in sequenza.
  - `learning_rate` (default 0.1): quanto conta ogni nuovo albero. Valori piccoli → apprendimento più lento e prudente, servono più alberi. `learning_rate` e `n_estimators` vanno scelti insieme.
  - `max_depth` (default 3): alberi piccoli.
  - `subsample` < 1.0: ogni albero usa solo una frazione dei campioni → casualità che riduce l'overfitting.
- **HistGradientBoosting:** versione moderna e molto più veloce su dataset grandi; gestisce direttamente i valori mancanti.

**Importanza delle feature**
- `feature_importances_`: per i modelli ad alberi, quanto ogni feature ha ridotto l'impurità in tutti gli split; i valori sommano a 1.
- Attenzione: favorisce le feature con molti valori diversi. Un'alternativa più affidabile è `permutation_importance` (`sklearn.inspection`): mescola una colonna alla volta e misura quanto peggiora il modello.
- Le importanze si possono usare per selezionare le feature (`SelectFromModel` in `sklearn.feature_selection`).

| | Bagging / Random Forest | Boosting |
|---|---|---|
| Addestramento | in parallelo, indipendente | in sequenza, ogni modello corregge il precedente |
| Riduce soprattutto | varianza | bias |
| Modelli base | alberi profondi | alberi piccoli |
| Rischio overfitting | basso | presente se troppi alberi / learning_rate alto |

## Esempi teorici di utilizzo

- **Dati tabellari aziendali** (clienti, transazioni, sensori): Random Forest e Gradient Boosting sono spesso i modelli più accurati, e richiedono poco preprocessing (niente scaling).
- **Primo modello "robusto":** un Random Forest con i parametri di default è un'ottima base di partenza su quasi ogni problema tabellare.
- **Competizioni e sistemi di scoring:** le versioni del gradient boosting (anche librerie esterne come XGBoost e LightGBM) sono tra le più usate.
- **Capire quali fattori contano:** un'azienda manifatturiera usa le importanze per scoprire quali parametri di produzione influenzano di più i difetti.

## Esempi di codice

### Base
Cosa mostra: un Random Forest è più accurato e più stabile di un singolo albero.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

X, y = load_breast_cancer(return_X_y=True)
acc_albero, acc_foresta = [], []
for seme in range(5):                                       # 5 split diversi degli stessi dati
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=seme, stratify=y)
    acc_albero.append(DecisionTreeClassifier(random_state=0).fit(X_train, y_train).score(X_test, y_test))
    acc_foresta.append(RandomForestClassifier(n_estimators=100, random_state=0)
                       .fit(X_train, y_train).score(X_test, y_test))

print("Albero:  media", round(np.mean(acc_albero), 3), "dev. std", round(np.std(acc_albero), 3))
print("Foresta: media", round(np.mean(acc_foresta), 3), "dev. std", round(np.std(acc_foresta), 3))

- La foresta ha una media più alta (circa 0.96 contro 0.93) e una deviazione standard più bassa: risultati più stabili tra uno split e l'altro.
- Il codice cambia di una sola riga: il Random Forest ha la stessa API dell'albero.
- Non serve scaling: la foresta è fatta di alberi.

### Intermedio
Cosa mostra: importanza delle feature e stima out-of-bag di un Random Forest.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

data = load_wine()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42, stratify=data.target)

rf = RandomForestClassifier(n_estimators=200, oob_score=True, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
print("Stima out-of-bag:", round(rf.oob_score_, 3))          # validazione "gratuita" sul training
print("Accuratezza test:", round(rf.score(X_test, y_test), 3))

importanze = pd.Series(rf.feature_importances_, index=data.feature_names).sort_values()
importanze.plot.barh(figsize=(7, 5), title="Importanza delle feature - Random Forest")
plt.tight_layout(); plt.show()
print(importanze.tail(3).round(3))                            # le 3 feature più importanti

- `oob_score_` è calcolato solo con i campioni che ogni albero non ha visto: è una stima simile a quella del test, senza usare il test.
- Il grafico a barre (richiesto anche nell'esercizio Wine del PDF) mostra le feature che guidano le decisioni.
- Le importanze sommano a 1: si leggono come "quota di merito" di ogni feature.

### Avanzato
Cosa mostra: confronto tra modelli di boosting e l'interazione tra learning_rate e n_estimators.

In [ ]:
import time
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import (AdaBoostClassifier, GradientBoostingClassifier,
                              HistGradientBoostingClassifier)

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

modelli = {
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "GB lr=0.1, 100 alberi": GradientBoostingClassifier(random_state=42),
    "GB lr=0.01, 100 alberi": GradientBoostingClassifier(learning_rate=0.01, random_state=42),
    "GB lr=0.01, 1000 alberi": GradientBoostingClassifier(learning_rate=0.01, n_estimators=1000, random_state=42),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=42),
}
for nome, m in modelli.items():
    inizio = time.perf_counter()
    m.fit(X_train, y_train)
    durata = time.perf_counter() - inizio
    print(f"{nome:25s} test={m.score(X_test, y_test):.3f}  tempo={durata:.2f}s")

- Con `learning_rate=0.01` e solo 100 alberi il modello non ha "fatto abbastanza strada": accuratezza più bassa.
- Con lo stesso learning_rate ma 1000 alberi recupera, ma impiega circa 10 volte il tempo.
- `HistGradientBoosting` raggiunge risultati pari o migliori in meno tempo: è la scelta consigliata su dataset grandi.
- `time.perf_counter()` misura il tempo trascorso in secondi.

## Esercizi

### Facile
Obiettivo: su `load_digits()` confronta un `DecisionTreeClassifier` e un `RandomForestClassifier` (split 80/20 stratificato). Poi prova `n_estimators` = 1, 10, 50, 200 e stampa l'accuratezza di test per ciascuno.

In [ ]:
# Scrivi qui la tua soluzione


### Medio
Obiettivo: su `load_breast_cancer()` addestra un Random Forest e mostra le 10 feature più importanti secondo `feature_importances_` e secondo `permutation_importance` (calcolata sul test, `n_repeats=10`). Le due classifiche coincidono? Disegnale affiancate.

In [ ]:
# Scrivi qui la tua soluzione


### Difficile
Obiettivo: su `load_diabetes()` (regressione) confronta `RandomForestRegressor` e `GradientBoostingRegressor`. Per il Gradient Boosting usa `staged_predict` per calcolare l'R² sul test dopo ogni albero (fino a 500, `learning_rate` 0.1 e 0.02) e disegna le due curve: dopo quanti alberi il modello smette di migliorare o inizia a peggiorare?

In [ ]:
# Scrivi qui la tua soluzione


# 4.2 Parametri, iperparametri e compromesso bias-varianza

## Spiegazione teorica

**Ripresa dal Giorno 1**
- **Parametri:** valori che il modello **impara da solo** durante `fit` (coefficienti di una regressione, soglie e struttura di un albero, pesi di una rete neurale). In scikit-learn: attributi con `_` finale.
- **Iperparametri:** "manopole" **esterne** che si impostano **prima** del `fit` e decidono *come* il modello impara. Non vengono appresi dai dati: vanno scelti da noi o da un algoritmo di ricerca.

**Gli iperparametri visti finora e cosa controllano**
| Modello | Iperparametro | Più complesso quando… |
|---|---|---|
| KNN | `n_neighbors` | k **diminuisce** |
| Albero | `max_depth`, `min_samples_leaf` | profondità aumenta / foglie più piccole |
| Ridge / Lasso | `alpha` | alpha **diminuisce** |
| Logistica / SVM | `C` | C **aumenta** |
| SVM rbf | `gamma` | gamma aumenta |
| Gradient Boosting | `n_estimators`, `learning_rate`, `max_depth` | più alberi, learning rate più alto, alberi più profondi |

**Compromesso bias-varianza**
- L'errore di un modello su dati nuovi ha tre componenti:
  - **bias:** errore dovuto a ipotesi troppo semplici (il modello non può rappresentare la realtà) → underfitting;
  - **varianza:** errore dovuto alla sensibilità ai dati specifici di training (cambiando i dati, cambia molto il modello) → overfitting;
  - **rumore irriducibile:** la parte casuale dei dati, che nessun modello può eliminare.
- Aumentando la complessità il bias scende e la varianza sale: l'errore totale ha un **minimo** nel mezzo. Gli iperparametri servono a trovare quel punto.

**Due strumenti di diagnosi**
- **Curva di validazione:** punteggio su training e validation al variare di **un iperparametro**. Mostra dove inizia l'overfitting (lo abbiamo fatto a mano con k nel Giorno 1).
- **Curva di apprendimento (learning curve):** punteggio su training e validation al variare del **numero di campioni** di training.
  - Curve vicine ma entrambe basse → **alto bias**: più dati non aiutano, serve un modello più complesso o feature migliori.
  - Grande distanza tra training (alto) e validation (basso) che si riduce con più dati → **alta varianza**: raccogliere più dati o regolarizzare aiuta.

**Metodi utili**
- `modello.get_params()`: dizionario di tutti gli iperparametri con i loro valori.
- `modello.set_params(max_depth=3)`: modifica un iperparametro senza ricreare l'oggetto.

## Esempi teorici di utilizzo

- **Diagnosi prima di investire:** prima di spendere soldi per raccogliere nuovi dati, la learning curve dice se servirebbero davvero.
- **Scelta della complessità:** la curva di validazione indica la profondità massima oltre la quale un albero smette di migliorare.
- **Documentazione del modello:** in un report si elencano gli iperparametri scelti (`get_params`) perché il modello sia riproducibile.
- **Comunicazione:** "il modello ha alto bias" si traduce per il cliente in "mancano informazioni nei dati", "alta varianza" in "servono più esempi o un modello più prudente".

## Esempi di codice

### Base
Cosa mostra: distinguere concretamente iperparametri e parametri appresi.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier

X, y = load_iris(return_X_y=True)
albero = DecisionTreeClassifier(max_depth=2, random_state=0)

print("Iperparametri:", {k: v for k, v in albero.get_params().items()
                         if k in ["max_depth", "criterion", "min_samples_leaf"]})

albero.fit(X, y)
print("Parametri appresi: profondità reale =", albero.get_depth(),
      "| foglie =", albero.get_n_leaves(), "| classi =", albero.classes_)

albero.set_params(max_depth=4).fit(X, y)                  # cambio la "manopola" e riaddestro
print("Dopo set_params: foglie =", albero.get_n_leaves())

- `get_params()` restituisce ciò che abbiamo deciso noi (o i valori di default).
- Struttura dell'albero, numero di foglie e classi esistono solo dopo il `fit`: sono appresi.
- `set_params` è lo stesso meccanismo che usano internamente gli strumenti di ricerca degli iperparametri.

### Intermedio
Cosa mostra: una curva di validazione costruita a mano con un validation set.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X, y = load_breast_cancer(return_X_y=True)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

profondita = range(1, 16)
train_sc, val_sc = [], []
for d in profondita:
    m = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_train, y_train)
    train_sc.append(m.score(X_train, y_train))
    val_sc.append(m.score(X_val, y_val))

plt.plot(profondita, train_sc, "o-", label="training")
plt.plot(profondita, val_sc, "o-", label="validation")
plt.xlabel("max_depth (complessità →)"); plt.ylabel("accuratezza"); plt.legend()
plt.title("Curva di validazione"); plt.show()
print("Profondità migliore sul validation:", profondita[val_sc.index(max(val_sc))])

- A sinistra (alberi bassi) entrambe le curve sono più basse: bias alto.
- Oltre una certa profondità il training arriva a 1.0 mentre il validation smette di migliorare: varianza alta.
- Il punto migliore è dove la curva di validation è massima, non dove lo è quella di training.

### Avanzato
Cosa mostra: curve di apprendimento per riconoscere un modello ad alto bias e uno ad alta varianza.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X, y = load_digits(return_X_y=True)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

modelli = {"Albero max_depth=2 (alto bias)": DecisionTreeClassifier(max_depth=2, random_state=0),
           "Albero senza limiti (alta varianza)": DecisionTreeClassifier(random_state=0)}
dimensioni = np.linspace(0.1, 1.0, 8)                       # dal 10% al 100% del training

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (nome, m) in zip(axes, modelli.items()):
    tr, va = [], []
    for frazione in dimensioni:
        n = int(frazione * len(X_train))
        m.fit(X_train[:n], y_train[:n])                      # addestra su una parte crescente dei dati
        tr.append(m.score(X_train[:n], y_train[:n]))
        va.append(m.score(X_val, y_val))
    ax.plot(dimensioni * len(X_train), tr, "o-", label="training")
    ax.plot(dimensioni * len(X_train), va, "o-", label="validation")
    ax.set_title(nome); ax.set_xlabel("campioni di training"); ax.legend()
axes[0].set_ylabel("accuratezza"); plt.show()

- Albero con profondità 2: le due curve si avvicinano subito e restano basse → aggiungere dati non serve.
- Albero senza limiti: training sempre 1.0, validation molto più bassa ma in crescita → più dati aiutano.
- scikit-learn offre `learning_curve` e `validation_curve` (`sklearn.model_selection`) che fanno lo stesso usando la cross-validation (modulo 4.3).

## Esercizi

### Facile
Obiettivo: crea una `SVC()`, un `RandomForestClassifier()` e un `KNeighborsClassifier()` e, per ciascuno, stampa con `get_params()` il valore di default di 3 iperparametri visti nel corso. Poi addestrali su `load_wine()` e stampa almeno un parametro appreso per ciascuno.

In [ ]:
# Scrivi qui la tua soluzione


### Medio
Obiettivo: su `load_digits()` (con scaling corretto) costruisci a mano la curva di validazione di una `SVC` al variare di `gamma` in `np.logspace(-5, 0, 10)` (asse x logaritmico) e indica la zona di underfitting, quella di overfitting e il valore migliore.

In [ ]:
# Scrivi qui la tua soluzione


### Difficile
Obiettivo: usa `learning_curve` di scikit-learn (`cv=5`, `train_sizes=np.linspace(0.1, 1.0, 8)`) su `load_breast_cancer()` per una regressione logistica con `C=0.001` e una con `C=1000` (entrambe con scaling: usa `make_pipeline(StandardScaler(), LogisticRegression(...))`). Disegna media e deviazione standard delle curve (con `fill_between`) e scrivi una diagnosi bias/varianza per ciascun modello.

In [ ]:
# Scrivi qui la tua soluzione


# 4.3 Cross-validation (validazione incrociata)

## Spiegazione teorica

**Il problema di un singolo split**
- Con un solo train/test split il risultato dipende da **quali** campioni sono finiti nel test: con un altro `random_state` l'accuratezza può cambiare di diversi punti (lo abbiamo visto nel Giorno 1 e nell'esempio Base del modulo 4.1).
- Con pochi dati, inoltre, tenere da parte training, validation **e** test lascia poco per addestrare.

**K-Fold Cross-Validation**
1. Si dividono i dati in **k parti** uguali, chiamate **fold** (pieghe).
2. Si addestra il modello k volte: ogni volta un fold diverso fa da validation e gli altri k−1 fold da training.
3. Si ottengono **k punteggi**: la **media** è la stima delle prestazioni, la **deviazione standard** dice quanto è stabile.
- Ogni campione viene usato **esattamente una volta** per la validazione e k−1 volte per l'addestramento.
- k tipico: 5 o 10. Costo: k addestramenti invece di uno.

```
Fold:      1      2      3      4      5
Giro 1:  [VAL] [TRN] [TRN] [TRN] [TRN]
Giro 2:  [TRN] [VAL] [TRN] [TRN] [TRN]
Giro 3:  [TRN] [TRN] [VAL] [TRN] [TRN]
Giro 4:  [TRN] [TRN] [TRN] [VAL] [TRN]
Giro 5:  [TRN] [TRN] [TRN] [TRN] [VAL]
```

**Varianti**
- `KFold(n_splits, shuffle, random_state)`: divisione semplice. Attenzione: di default **non mescola** (`shuffle=False`); se i dati sono ordinati per classe il risultato è disastroso.
- `StratifiedKFold`: mantiene in ogni fold le **proporzioni delle classi** → la scelta standard per la classificazione, indispensabile con classi sbilanciate.
- `LeaveOneOut`: k = numero di campioni (un solo campione in validation per volta); solo per dataset piccolissimi.
- `TimeSeriesSplit`: per dati temporali, valida sempre su periodi **successivi** a quelli di training (evita il leakage temporale).

**Funzioni di scikit-learn**
- `cross_val_score(modello, X, y, cv=5, scoring="accuracy", n_jobs=-1)` → array con k punteggi.
  - `cv` intero: per i classificatori usa automaticamente `StratifiedKFold`, per i regressori `KFold` (in entrambi i casi senza shuffle).
  - `cv` può essere anche un oggetto (`StratifiedKFold(5, shuffle=True, random_state=42)`) per avere pieno controllo.
- `cross_validate(...)`: come sopra ma con **più metriche** insieme (`scoring=["accuracy", "f1"]`), tempi e, con `return_train_score=True`, anche i punteggi di training.
- `scoring`: stringa con la metrica. Esempi: `"accuracy"`, `"f1"`, `"f1_macro"`, `"recall"`, `"r2"`, `"neg_mean_absolute_error"`, `"neg_root_mean_squared_error"`.
  - Le metriche di errore sono **negative** perché scikit-learn segue la convenzione "più alto è meglio": −40 è meglio di −50.

**Dove si colloca nel workflow**
- Si tiene comunque da parte un **test set finale**.
- Sul resto dei dati la cross-validation sostituisce il validation set singolo: serve a confrontare modelli e scegliere gli iperparametri.
- Attenzione al leakage: se lo scaler viene addestrato su tutti i dati prima della cross-validation, ogni fold di validazione "è stato visto". La soluzione è la **Pipeline** (modulo 4.5).

## Esempi teorici di utilizzo

- **Dataset piccoli** (poche centinaia di righe, comuni in ambito medico): la cross-validation usa ogni dato sia per addestrare sia per valutare.
- **Confronto tra modelli:** "Random Forest 0.96 ± 0.01 contro KNN 0.93 ± 0.03" è molto più affidabile di due numeri presi da un solo split.
- **Classi sbilanciate (frodi, guasti):** StratifiedKFold garantisce che ogni fold contenga qualche esempio della classe rara.
- **Previsioni di vendita:** con dati temporali si usa TimeSeriesSplit, perché non si può addestrare sul futuro per prevedere il passato.

## Esempi di codice

### Base
Cosa mostra: come K-Fold divide i dati e come cross_val_score fa tutto in una riga.

In [ ]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import KFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier

X, y = load_wine(return_X_y=True)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for i, (idx_train, idx_val) in enumerate(kf.split(X), start=1):   # indici delle righe per ogni giro
    print(f"Giro {i}: {len(idx_train)} righe di training, {len(idx_val)} di validation, "
          f"prime righe di validation: {idx_val[:5]}")

punteggi = cross_val_score(DecisionTreeClassifier(random_state=0), X, y, cv=kf)
print("Punteggi:", punteggi.round(3))
print(f"Media: {punteggi.mean():.3f} ± {punteggi.std():.3f}")

- `kf.split(X)` restituisce, per ogni giro, gli indici delle righe di training e di validation.
- `cross_val_score` esegue internamente lo stesso ciclo: k volte fit + score.
- Il risultato si comunica come "media ± deviazione standard".

### Intermedio
Cosa mostra: perché lo shuffle e la stratificazione sono importanti.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier

X, y = load_iris(return_X_y=True)
print(y[:5], y[50:55], y[100:105])           # il dataset è ORDINATO per classe: 50 + 50 + 50

modello = DecisionTreeClassifier(random_state=0)
senza_shuffle = cross_val_score(modello, X, y, cv=KFold(n_splits=3))          # shuffle=False di default
stratificato = cross_val_score(modello, X, y, cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=0))

print("KFold senza shuffle:   ", senza_shuffle)     # [0. 0. 0.]
print("StratifiedKFold:       ", stratificato.round(3))

- Senza shuffle ogni fold contiene **una sola specie**: il modello si addestra su due specie e viene valutato sulla terza, che non ha mai visto → accuratezza 0.
- StratifiedKFold mette in ogni fold un terzo di ogni specie: la valutazione è sensata.
- Usando `cv=3` (intero) con un classificatore scikit-learn usa già StratifiedKFold: il problema nasce quando si sceglie KFold a mano.

### Avanzato
Cosa mostra: confrontare più modelli con più metriche usando cross_validate.

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

X, y = load_breast_cancer(return_X_y=True)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

modelli = {"Albero": DecisionTreeClassifier(random_state=0),
           "Random Forest": RandomForestClassifier(random_state=0),
           "Gradient Boosting": GradientBoostingClassifier(random_state=0)}

righe = []
for nome, m in modelli.items():
    ris = cross_validate(m, X, y, cv=cv, scoring=["accuracy", "f1"], return_train_score=True, n_jobs=-1)
    righe.append({"modello": nome,
                  "train acc": ris["train_accuracy"].mean(),
                  "val acc": ris["test_accuracy"].mean(),       # "test_" = fold di validazione
                  "val acc std": ris["test_accuracy"].std(),
                  "val f1": ris["test_f1"].mean(),
                  "tempo fit (s)": ris["fit_time"].mean()})
print(pd.DataFrame(righe).round(3))

- `cross_validate` restituisce un dizionario con una chiave per metrica (`test_accuracy`, `test_f1`, …) e i tempi.
- Le chiavi si chiamano `test_…` ma si riferiscono ai fold di validazione, non al test set finale.
- Tutti e tre arrivano a 1.0 sul training: è la validation a distinguerli, e l'albero singolo perde circa 4 punti rispetto agli ensemble.
- La colonna `val acc std` mostra anche quale modello è più stabile tra i fold.
- Qui non serve scaling perché sono tutti modelli ad alberi; con KNN o SVM servirebbe una Pipeline per evitare il leakage.

## Esercizi

### Facile
Obiettivo: su `load_digits()` calcola con `cross_val_score` (cv=5) l'accuratezza di un Random Forest e stampa i 5 punteggi, la media e la deviazione standard. Ripeti con cv=10: cosa cambia?

In [ ]:
# Scrivi qui la tua soluzione


### Medio
Obiettivo: su `load_diabetes()` confronta con `cross_val_score` e `KFold(5, shuffle=True, random_state=0)` una `LinearRegression`, un `Ridge(alpha=1)` e un `RandomForestRegressor`, usando `scoring="r2"` e poi `scoring="neg_mean_absolute_error"`. Stampa il MAE come numero positivo e spiega perché scikit-learn lo restituisce negativo.

In [ ]:
# Scrivi qui la tua soluzione


### Difficile
Obiettivo: crea un dataset sbilanciato con `make_classification(n_samples=300, weights=[0.95, 0.05], random_state=1)`. Con `KFold(5, shuffle=True)` e `StratifiedKFold(5, shuffle=True)` conta, per ogni fold, quanti campioni della classe rara finiscono in validation, e confronta i punteggi F1 ottenuti da un Random Forest con i due metodi. Ripeti per 10 valori di `random_state` e confronta la deviazione standard delle stime.

In [ ]:
# Scrivi qui la tua soluzione


# 4.4 Ricerca degli iperparametri: GridSearchCV e RandomizedSearchCV

## Spiegazione teorica

**Il problema:** ogni modello ha più iperparametri che interagiscono tra loro (C e gamma della SVM, learning_rate e n_estimators del boosting). Provarli a mano è lento e soggetto a errori.

**GridSearchCV — ricerca a griglia**
- Si definisce una **griglia**: un dizionario con, per ogni iperparametro, la lista di valori da provare.
  `param_grid = {"max_depth": [3, 5, 7], "criterion": ["gini", "entropy"]}`
- Si provano **tutte le combinazioni** (qui 3 × 2 = 6) e ognuna viene valutata con la **cross-validation** (es. 5 fold → 30 addestramenti).
- Si sceglie la combinazione con il punteggio medio di cross-validation più alto.
- Con `refit=True` (default) il modello migliore viene **riaddestrato su tutto il training**, pronto per `predict`.
- La griglia può anche essere una **lista di dizionari**, per provare combinazioni diverse per casi diversi (es. `gamma` solo quando `kernel="rbf"`).

**Attributi dopo il `fit`**
- `best_params_`: la combinazione migliore.
- `best_score_`: il suo punteggio **medio di cross-validation** (non è il punteggio sul test).
- `best_estimator_`: il modello migliore già riaddestrato.
- `cv_results_`: tutti i risultati (convertibile in DataFrame).
- L'oggetto si usa direttamente come un modello: `grid.predict(X_test)`, `grid.score(X_test, y_test)`.

**Parametri principali**
- `estimator`: il modello da ottimizzare; `param_grid`: la griglia; `cv`: numero di fold o oggetto di cross-validation; `scoring`: la metrica da massimizzare; `n_jobs=-1`: usa tutti i processori; `verbose`: stampa l'avanzamento.

**Limite:** il numero di combinazioni **moltiplica**: 5 iperparametri con 5 valori ciascuno sono 3125 combinazioni × 5 fold = 15.625 addestramenti.

**RandomizedSearchCV — ricerca casuale**
- Invece di provare tutto, estrae a caso **`n_iter` combinazioni** da liste o da **distribuzioni** di probabilità.
- Vantaggi: il costo è fissato da `n_iter`; esplora più valori diversi per ogni iperparametro; di solito trova soluzioni vicine a quelle della griglia in molto meno tempo.
- Distribuzioni utili (`scipy.stats`):
  - `randint(a, b)`: interi da a a **b−1** (es. numero di alberi);
  - `uniform(loc, scale)`: decimali da `loc` a **`loc + scale`** (attenzione: `uniform(0.1, 10)` va da 0.1 a 10.1, non a 10, come invece indica il commento nel PDF);
  - `loguniform(a, b)`: decimali tra a e b distribuiti in modo uniforme sugli **ordini di grandezza** (0.001, 0.01, 0.1…), ideale per C, gamma, alpha, learning_rate.
- `random_state` rende riproducibile l'estrazione.

**Regole pratiche**
- Il test set resta **fuori** dalla ricerca: si usa solo alla fine, sul `best_estimator_`.
- Se servono preprocessing (scaling, imputazione), vanno messi in una **Pipeline** dentro la ricerca, altrimenti si ha leakage tra i fold (modulo 4.5).
- Strategia comune: prima una ricerca casuale ampia, poi una griglia fine attorno ai valori migliori.

## Esempi teorici di utilizzo

- **SVM:** C e gamma vanno scelti insieme (lo abbiamo visto nella tabella del Giorno 3): una griglia logaritmica li esplora in modo sistematico.
- **Gradient Boosting con molti iperparametri:** una ricerca casuale con 50 iterazioni esplora uno spazio che a griglia richiederebbe migliaia di combinazioni.
- **Ottimizzare la metrica giusta:** in un sistema antifrode si imposta `scoring="recall"` o `"f1"` invece dell'accuratezza.
- **Riproducibilità e report:** `cv_results_` documenta tutte le prove fatte, utile per giustificare le scelte al cliente.

## Esempi di codice

### Base
Cosa mostra: GridSearchCV su un albero di decisione (l'esempio del PDF, completato con la valutazione finale).

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier

X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

param_grid = {"max_depth": [2, 3, 5, 7, None],          # 5 valori
              "criterion": ["gini", "entropy"]}          # × 2 valori = 10 combinazioni
grid = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5)
grid.fit(X_train, y_train)                               # 10 combinazioni × 5 fold = 50 addestramenti

print("Migliori iperparametri:", grid.best_params_)
print("Punteggio medio in CV:", round(grid.best_score_, 3))
print("Accuratezza sul test:", round(grid.score(X_test, y_test), 3))   # usa il best_estimator_

- La cross-validation avviene solo sul training; il test serve per la verifica finale.
- `grid` dopo il `fit` si comporta come il modello migliore: `score` e `predict` usano `best_estimator_`.
- `best_score_` e accuratezza sul test sono due numeri diversi: il primo guida la scelta, il secondo la conferma.

### Intermedio
Cosa mostra: leggere tutti i risultati di una griglia e visualizzarli come tabella.

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

param_grid = {"n_estimators": [50, 100, 200], "max_depth": [3, 6, None]}
grid = GridSearchCV(RandomForestClassifier(random_state=0), param_grid,
                    cv=5, scoring="f1", n_jobs=-1)        # ottimizziamo l'F1, non l'accuratezza
grid.fit(X_train, y_train)

pd.set_option("display.width", 150); pd.set_option("display.max_columns", None)
risultati = pd.DataFrame(grid.cv_results_)
risultati["param_max_depth"] = risultati["param_max_depth"].fillna("None").astype(str)   # None diventa leggibile
colonne = ["param_n_estimators", "param_max_depth", "mean_test_score", "std_test_score", "rank_test_score"]
print(risultati[colonne].sort_values("rank_test_score").round(3).head())
print(risultati.pivot_table(index="param_max_depth", columns="param_n_estimators",
                            values="mean_test_score").round(3))   # tabella max_depth × n_estimators

- `cv_results_` contiene una riga per combinazione con media, deviazione standard e posizione in classifica.
- Spesso le prime combinazioni sono molto vicine: conviene scegliere la più semplice tra quelle equivalenti.
- `scoring="f1"` fa sì che `best_params_` sia la combinazione con l'F1 medio più alto.

### Avanzato
Cosa mostra: RandomizedSearchCV con distribuzioni, confrontato con una griglia di costo simile.

In [ ]:
import time
from scipy.stats import randint, uniform, loguniform
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

distribuzioni = {
    "n_estimators": randint(50, 300),            # interi da 50 a 299
    "learning_rate": loguniform(0.01, 0.3),      # uniforme sugli ordini di grandezza
    "max_depth": randint(1, 5),                  # 1, 2, 3 o 4
    "subsample": uniform(0.6, 0.4),              # da 0.6 a 1.0 (loc, loc + scale)
}
ricerca = RandomizedSearchCV(GradientBoostingClassifier(random_state=0), distribuzioni,
                             n_iter=20, cv=5, random_state=42, n_jobs=-1)
inizio = time.perf_counter(); ricerca.fit(X_train, y_train)
print(f"Random: {time.perf_counter() - inizio:.1f}s  CV={ricerca.best_score_:.3f}  "
      f"test={ricerca.score(X_test, y_test):.3f}")
print("Migliori:", {k: round(float(v), 3) for k, v in ricerca.best_params_.items()})

griglia = {"n_estimators": [100, 200], "learning_rate": [0.05, 0.1],
           "max_depth": [2, 3], "subsample": [0.8, 1.0]}                  # 16 combinazioni
grid = GridSearchCV(GradientBoostingClassifier(random_state=0), griglia, cv=5, n_jobs=-1)
inizio = time.perf_counter(); grid.fit(X_train, y_train)
print(f"Grid:   {time.perf_counter() - inizio:.1f}s  CV={grid.best_score_:.3f}  "
      f"test={grid.score(X_test, y_test):.3f}")

- `n_iter=20` fissa il costo: 20 combinazioni × 5 fold = 100 addestramenti, qualunque sia l'ampiezza dello spazio.
- La griglia prova solo 2 valori per iperparametro; la ricerca casuale ne esplora 20 diversi per ciascuno.
- I due approcci arrivano a risultati simili con un costo simile: la differenza diventa decisiva quando gli iperparametri sono molti (la griglia esplode, la ricerca casuale no).
- `loguniform` evita di sprecare tentativi: con `uniform(0.01, 0.3)` quasi tutti i valori estratti sarebbero sopra 0.03.

## Esercizi

### Facile
Obiettivo: su `load_iris()` usa `GridSearchCV` per scegliere `n_neighbors` di un KNN tra 1 e 30 (cv=5). Stampa `best_params_`, `best_score_` e disegna il `mean_test_score` di `cv_results_` in funzione di k.

In [ ]:
# Scrivi qui la tua soluzione


### Medio
Obiettivo: su `load_wine()` ottimizza un `RandomForestClassifier` (esercizio del PDF) con una griglia su `n_estimators` [50, 100, 200], `max_depth` [None, 5, 10] e `criterion` ["gini", "entropy"], con `cv=5`. Valuta il modello migliore sul test con `classification_report` e matrice di confusione, e mostra le 5 feature più importanti del `best_estimator_`.

In [ ]:
# Scrivi qui la tua soluzione


### Difficile
Obiettivo: su `load_digits()` ottimizza un `HistGradientBoostingClassifier` con `RandomizedSearchCV` (`n_iter=25`) su `learning_rate` (loguniform 0.01–0.5), `max_iter` (randint 50–400), `max_leaf_nodes` (randint 10–60) e `l2_regularization` (loguniform 1e-4–1). Poi fai una seconda `GridSearchCV` fine attorno ai valori migliori trovati. Confronta tempo, `best_score_` e accuratezza sul test delle due fasi.

In [ ]:
# Scrivi qui la tua soluzione


# 4.5 Pipeline

## Spiegazione teorica

**Il problema**
- Un flusso reale ha più passaggi: imputazione → codifica → scaling → modello. Ogni passaggio va addestrato sul training e applicato al test, nello stesso ordine.
- Farlo a mano è lungo e facile da sbagliare. Dentro la cross-validation è ancora peggio: lo scaler dovrebbe essere riaddestrato **in ogni fold** usando solo la parte di training di quel fold, altrimenti c'è leakage.

**La soluzione: `Pipeline`**
- Una Pipeline concatena più passaggi in **un unico oggetto** che si comporta come un modello.
- Sintassi: una lista di tuple `(nome, oggetto)`:
  `Pipeline([("scaler", StandardScaler()), ("svc", SVC())])`
  - tutti i passaggi tranne l'ultimo devono essere **transformer** (avere `fit` e `transform`);
  - l'ultimo è il **modello** (estimator);
  - i nomi devono essere **unici** e non contenere il doppio underscore `__`.
- `make_pipeline(StandardScaler(), SVC())` fa lo stesso assegnando i nomi in automatico (il nome della classe in minuscolo: `"standardscaler"`, `"svc"`).

**Cosa succede quando si chiama…**
- `pipeline.fit(X_train, y_train)`: per ogni transformer esegue `fit_transform` e passa il risultato al successivo; alla fine esegue `fit` del modello.
- `pipeline.predict(X_test)`: per ogni transformer esegue **solo `transform`** (con le statistiche imparate sul training), poi `predict` del modello.
- La regola d'oro del Giorno 2 viene rispettata **automaticamente**.

**Pipeline + cross-validation / ricerca degli iperparametri**
- Passando la Pipeline a `cross_val_score` o `GridSearchCV`, in ogni fold l'**intera** catena viene riaddestrata solo sui dati di training del fold: niente leakage.
- Per indicare un iperparametro di un passaggio si usa la sintassi **`nomepassaggio__parametro`** (doppio underscore):
  `param_grid = {"svc__C": [0.1, 1, 10], "svc__kernel": ["linear", "rbf"]}`
- Con strutture annidate (ColumnTransformer dentro la Pipeline) i nomi si concatenano: `"prep__num__imputer__strategy"`.
- Si possono ottimizzare anche le scelte di preprocessing (es. la strategia di imputazione o il numero di componenti PCA del Giorno 5).

**Accesso ai passaggi**
- `pipeline.named_steps["svc"]` oppure `pipeline["svc"]`: il singolo oggetto addestrato (es. per leggere `coef_` o `feature_importances_`).
- `pipeline[:-1]`: tutti i passaggi tranne il modello (utile per vedere i dati trasformati).

**Salvare e ricaricare il modello completo**
- `joblib.dump(pipeline, "modello.joblib")` salva preprocessing + modello in un file; `joblib.load("modello.joblib")` lo ricarica pronto per `predict`.
- Salvando la Pipeline intera non si rischia di dimenticare lo scaler o di applicare un preprocessing diverso in produzione.

## Esempi teorici di utilizzo

- **Messa in produzione:** un solo file contiene tutto il necessario per trasformare i dati grezzi in previsioni.
- **Cross-validation corretta con KNN, SVM, regressione logistica:** lo scaling avviene dentro ogni fold.
- **Ottimizzare preprocessing e modello insieme:** chiedere a GridSearchCV se conviene imputare con media o mediana, e con quale C.
- **Lavoro in team:** la Pipeline documenta esattamente la sequenza di trasformazioni usata.

## Esempi di codice

### Base
Cosa mostra: una Pipeline scaling + SVM usata come un normale modello.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipeline = Pipeline([
    ("scaler", StandardScaler()),              # transformer: fit_transform nel fit, transform nel predict
    ("svc", SVC(kernel="rbf", C=1)),           # modello finale
])
pipeline.fit(X_train, y_train)                 # scaler addestrato SOLO sul training, poi la SVM
print("Accuratezza test:", round(pipeline.score(X_test, y_test), 3))
print("Media imparata dallo scaler:", pipeline.named_steps["scaler"].mean_[:3].round(2))
print("Vettori di supporto:", pipeline["svc"].n_support_)

- Una sola chiamata a `fit` e una a `score`: lo scaling avviene internamente nel modo corretto.
- `named_steps` (o la notazione `pipeline["nome"]`) dà accesso ai singoli passaggi addestrati.
- Confronta con il modulo 2.1: stesso risultato, metà del codice e nessuna possibilità di dimenticare `transform` sul test.

### Intermedio
Cosa mostra: GridSearchCV su una Pipeline con la sintassi `passo__parametro`.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

pipeline = Pipeline([("scaler", StandardScaler()), ("svc", SVC())])
param_grid = [
    {"svc__kernel": ["linear"], "svc__C": [0.1, 1, 10]},                       # gamma non serve qui
    {"svc__kernel": ["rbf"], "svc__C": [1, 10, 100], "svc__gamma": [0.001, 0.01, 0.1]},
]
grid = GridSearchCV(pipeline, param_grid, cv=5, n_jobs=-1)
grid.fit(X_train, y_train)            # in OGNI fold lo scaler viene riaddestrato sui soli dati di training

print("Migliori:", grid.best_params_)
print("CV:", round(grid.best_score_, 3), "| test:", round(grid.score(X_test, y_test), 3))

- `svc__C` significa "il parametro `C` del passaggio chiamato `svc`".
- La griglia come lista di dizionari evita combinazioni inutili (gamma con kernel lineare): 3 + 9 = 12 combinazioni invece di 18.
- Questa è la versione corretta della ricerca C × gamma fatta a mano nel Giorno 3.

### Avanzato
Cosa mostra: una Pipeline completa per dati misti (numeri, categorie, mancanti), ottimizzata e salvata su file.

In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV

# Dataset sintetico: clienti di una palestra
rng = np.random.RandomState(0)
n = 400
df = pd.DataFrame({
    "eta": rng.randint(18, 70, n).astype(float),
    "ingressi_mese": rng.poisson(8, n).astype(float),
    "mesi_iscrizione": rng.randint(1, 60, n).astype(float),
    "abbonamento": rng.choice(["base", "plus", "premium"], n),
    "citta": rng.choice(["Roma", "Milano", "Napoli"], n),
})
prob = 1 / (1 + np.exp(0.8 * (df["ingressi_mese"] - 6) + 1.0 * (df["abbonamento"] == "premium")))
df["disdetta"] = (rng.rand(n) < prob).astype(int)                     # chi va poco disdice di più
df.loc[rng.choice(n, 30, replace=False), "eta"] = np.nan              # valori mancanti
df.loc[rng.choice(n, 20, replace=False), "citta"] = np.nan

X, y = df.drop(columns="disdetta"), df["disdetta"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

numeriche = ["eta", "ingressi_mese", "mesi_iscrizione"]
categoriche = ["abbonamento", "citta"]
prep = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer()), ("scaler", StandardScaler())]), numeriche),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categoriche),
])
modello = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=1000))])

param_grid = {"prep__num__imputer__strategy": ["mean", "median"],    # nomi annidati con __
              "clf__C": [0.01, 0.1, 1, 10]}
grid = GridSearchCV(modello, param_grid, cv=5, scoring="f1").fit(X_train, y_train)
print("Migliori:", grid.best_params_, "| F1 test:", round(grid.score(X_test, y_test), 3))

joblib.dump(grid.best_estimator_, "modello_palestra.joblib")         # salva preprocessing + modello
caricato = joblib.load("modello_palestra.joblib")
nuovo = pd.DataFrame([{"eta": np.nan, "ingressi_mese": 1, "mesi_iscrizione": 3,
                       "abbonamento": "base", "citta": "Torino"}])   # mancante + città mai vista
print("Probabilità di disdetta:", caricato.predict_proba(nuovo)[0, 1].round(2))

- Ogni gruppo di colonne ha la sua mini-pipeline; il ColumnTransformer le unisce, la Pipeline esterna aggiunge il modello.
- `prep__num__imputer__strategy` attraversa i livelli: passo `prep` → gruppo `num` → passo `imputer` → parametro `strategy`.
- Il modello salvato accetta dati **grezzi**, anche con valori mancanti e categorie nuove, perché contiene tutto il preprocessing.
- `grid.score` usa la metrica scelta in `scoring` (qui l'F1).

## Esercizi

### Facile
Obiettivo: su `load_breast_cancer()` crea con `make_pipeline` una pipeline `StandardScaler` + `KNeighborsClassifier`, valutala con `cross_val_score` (cv=5) e stampa i nomi automatici dei passaggi (`pipeline.steps`).

In [ ]:
# Scrivi qui la tua soluzione


### Medio
Obiettivo: su `load_wine()` costruisci una Pipeline `StandardScaler` → `SVC` e usa `GridSearchCV` per scegliere insieme `svc__C`, `svc__gamma` e anche lo scaler, provando `StandardScaler()` e `MinMaxScaler()` come valori del passo `scaler` (suggerimento: `{"scaler": [StandardScaler(), MinMaxScaler()], ...}`). Riporta la combinazione migliore e l'accuratezza sul test.

In [ ]:
# Scrivi qui la tua soluzione


### Difficile
Obiettivo: crea un DataFrame sintetico di 500 noleggi di auto (feature numeriche con mancanti, almeno due categoriche, target `danno_restituzione` 0/1 sbilanciato al 10%). Costruisci una Pipeline completa con `ColumnTransformer` e `RandomForestClassifier`, ottimizzala con `RandomizedSearchCV` (`StratifiedKFold`, `scoring="recall"`) includendo almeno un iperparametro del preprocessing, salvala con `joblib`, ricaricala in uno script separato e usala per prevedere 3 noleggi nuovi scritti a mano.

In [ ]:
# Scrivi qui la tua soluzione
